## DATABRICKS CAPSTONE PROJECTS
<br>**DATE: 2026-09-04**
<br>**TOPIC: Order Transformation in Silver Layer**


| # | Transformation Task |
|---|----------------------|
| 1 | Remove duplicate orders | 
| 2 | Validate customer IDs |
| 3 | Standardize status | 
| 4 | Validate order dates | 
| 5 | Reject or quarantine negative order amounts | 
| 6 | Identify cancelled orders |
| 7 | Identify returned orders|
| 8 | Identify completed orders |

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
try:
    orders_df = spark.read.table('ecommerce.bronze.orders')
    row_count = orders_df.count()
    print(f'Orders Table loaded from Bronze layer with {row_count} rows')

except Exception as e:
    raise e

In [0]:
try:
    orders_df_dedup = orders_df.dropDuplicates()
    row_count = orders_df_dedup.count()
    print(f"Order tables has {row_count} rows post de-duplication")
except Exception as e:
    raise e

In [0]:
try:
    # Ensure the customer id present in orders table are actually valid customer

    customer_df = spark.read.table("ecommerce.silver.customers")

    orders_cust_check = orders_df_dedup.select("customer_id").distinct()

    orders_cust_check = orders_cust_check.withColumnRenamed(
        "customer_id", "customer_id_ord"
    )

    orders_cust_check = orders_cust_check.join(
        customer_df,
        customer_df.customer_id == orders_cust_check.customer_id_ord,
        "left",
    )

    faulty_ids = orders_cust_check.filter(col("customer_id").isNull()).count()

    # Check if there are any faulty customer ids then replace the value with 0
    if faulty_ids > 0:
        print(f"Found {faulty_ids} faulty customer ids in orders table")
    else:
        pass

    orders_df_dedup = orders_df_dedup.withColumn(
        "customer_id", coalesce(col("customer_id"), lit('0'))
    )

except Exception as e:
    raise e

In [0]:
try:
    orders_df_dedup = orders_df_dedup.withColumn(
        "order_status", trim(initcap(col("order_status")))
    )
except Exception as e:
    raise e

In [0]:
try:
    date_cols = [k for k, v in orders_df_dedup.dtypes if v == "timestamp"]

    #Check if there are any null values in the date columns

    for c in date_cols:
        count_rows = orders_df_dedup.filter(col(c).isNull()).count()
        count_rows_future_dates = orders_df_dedup.filter(col(c)> current_timestamp()).count()
        if (count_rows > 0):
            raise_error(f'{count_rows} are found with Null dates')
        elif count_rows_future_dates > 0:
            raise_error(f'{count_rows_future_dates} are found with future dates')
        else:
            pass
except Exception as e:
    raise e

In [0]:
try:
    orders_df_dedup = orders_df_dedup.withColumn(
        "QuarentineRecords",
        when(col("total_amount") < 0, col("total_amount")).otherwise(lit(None)),
    )

    rejected_orders = orders_df_dedup.filter(col("QuarentineRecords").isNotNull())

    orders_df_dedup = orders_df_dedup.filter(col("QuarentineRecords").isNull())

    orders_df_dedup = orders_df_dedup.drop("QuarentineRecords")

except Exception as e:
    raise e

In [0]:
try:
    order_summary = orders_df_dedup.groupBy(col("order_status")).agg(
        count("*").alias("count_per_order_status")
    )

    ##CANCELLED ORDERS, RETURNED ORDERS, COMPLETED ORDERS

    orders_cancelled = orders_df_dedup.filter(col("order_status") == "Cancelled")
    orders_returned = orders_df_dedup.filter(col("order_status") == "Returned")
    orders_completed = orders_df_dedup.filter(col("order_status") == "Delivered")

    order_summary.display()
    print(f"Cancelled: {orders_cancelled.count()}")
    print(f"Returned: {orders_returned.count()}")
    print(f"Completed: {orders_completed.count()}")


except Exception as e:
    raise e

In [0]:
try:
    orders_df_dedup.write.saveAsTable("ecommerce.silver.orders", mode="OVERWRITE")
except Exception as e:
    raise e